# Rung 2.2 — fine-tune `omr_transformer` on strips_v2_2 (rhythm signs)

Step-by-step context: **`docs/COLAB.md`** in the repo. Before running:
1. Build the package locally: `sh scripts/make_colab_zip.sh` → `data/colab/tnc_rung2_colab.zip`.
2. Upload that one zip to your Google Drive into a folder named **`tnc`** (path: `MyDrive/tnc/tnc_rung2_colab.zip`).
3. Runtime → Change runtime type → a **GPU** (free: T4; Pro: L4 or A100).
4. Run the cells top to bottom. Do the **smoke cell** once before the full run.

Checkpoints stream to `MyDrive/tnc/rung22/` — a disconnected session loses minutes, not the run (use the resume cell).

In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (grants this notebook access to MyDrive; approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy the package Drive -> VM disk and unzip there. Training reads images from the FAST local
# disk; reading 18k PNGs through the Drive mount would starve the dataloader.
%%time
!cp /content/drive/MyDrive/tnc/tnc_rung2_colab.zip /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/tnc_rung2_colab.zip
!ls /content/tnc/src/vision/ && head -c 300 /content/tnc/data/synthetic/strips_v2_2/manifest.jsonl

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# SMOKE TEST (~3 min): 100 tiny steps end-to-end — data, tokenizer extension (+21 ids),
# augmentation workers, AMP, checkpoint write to Drive. Loss must FALL. Run once per new setup.
%cd /content/tnc
!python src/vision/train.py --out-dir /content/drive/MyDrive/tnc/rung22-shakeout \
    --limit-train 512 --limit-val 128 --max-steps 100 --eval-every 50 --save-every 50 \
    --num-workers 2

In [ ]:
# FULL RUN (fresh start). Timing at defaults (batch 8, 6000 steps ~ 2.9 epochs):
#   T4 ~2.5-4 h | L4 ~1.5-2 h | A100 ~45-75 min.
# On L4/A100 add:  --batch-size 16   (fits easily; ~5.8 epochs -- fine, augmentation keeps
# epochs fresh and <out>/best tracks the lowest val loss anyway).
%cd /content/tnc
!python src/vision/train.py --out-dir /content/drive/MyDrive/tnc/rung22 --num-workers 2

In [ ]:
# RESUME after a disconnect: re-run the setup cells above (GPU check .. pip), then this.
# Continues from MyDrive/tnc/rung22/last with optimizer/scheduler state intact.
%cd /content/tnc
!python src/vision/train.py --out-dir /content/drive/MyDrive/tnc/rung22 --num-workers 2 --resume

In [ ]:
# HEADLINE METRIC on the best checkpoint: per-class AEU accidental accuracy (+ SER, exact match).
# This is what decides Rung 2 (see docs/PHASE2.md section 5); val loss alone is not the judge.
%cd /content/tnc
!python src/vision/eval_omr.py --checkpoint /content/drive/MyDrive/tnc/rung22/best

## Reading the result

- **Judge on `eval_omr.py`'s per-class table** (it also appends to `rung2/best/eval.jsonl` on Drive).
  Strong per-class AEU accuracy → Rung 2 passes → next is Rung 3 (real photos, `docs/PIPELINE.md` §3).
  Weak accuracy after honest tries (LR 1e-5..5e-5, more steps) → the CRNN+CTC fallback conversation.
- Everything you need to keep lives on Drive: `rung2/best/` (weights for the ONNX export later),
  `rung2/metrics.jsonl` (loss curves), `rung2/best/eval.jsonl` (eval history).
- Disconnects are normal on Colab. Reconnect → rerun setup cells → the **resume** cell.